# Rust Toolchain Local Variable Names

Three progressive submissions:
1. **Constant placeholder** (sanity floor, scores near 0).
2. A **type-signature heuristic** (hand-coded; shown as the strongest
   measured non-trained adversary; not a valid final submission under the
   task Rules since it is not a trained model).
3. The **intended submission**: a trained character-level model (GPU) —
   dilated char-CNN encoder, learned attention over every occurrence of a
   placeholder, char-level GRU decoder.


In [ ]:
import csv
from pathlib import Path

# Robust data-path discovery (never hardcode one location).
DATA = next(p for p in (Path("/data"), Path("dataset/public"),
                        Path("/kaggle/input/rust-variable-naming/public"))
            if (p / "test.csv").exists())
WORK = Path("working")
WORK.mkdir(exist_ok=True)

def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

train = read_csv(DATA / "train.csv")
test = read_csv(DATA / "test.csv")
print("train:", len(train), "test:", len(test))

def write_submission(id_to_names: dict):
    with open(WORK / "submission.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "names"])
        for r in test:
            w.writerow([r["id"], id_to_names.get(r["id"], "")])


In [ ]:
# Submission 1: constant placeholder "x" at every slot (sanity floor).
write_submission({r["id"]: "|".join(["x"] * int(r["num_vars"])) for r in test})
print("wrote constant-placeholder baseline")


In [ ]:
# Submission 2: type-signature heuristic (reference only; NOT a valid final
# submission -- no trained parameters). For each VARi, look up the most
# frequent training-set name observed for that parameter's declared type;
# fall back to the most frequent name observed at that positional slot index.
import re
from collections import defaultdict, Counter

type_pat = re.compile(r"VAR(\d+)\s*:\s*([^,)]+)")

type_name_counts = defaultdict(Counter)
pos_name_counts = defaultdict(Counter)
for r in train:
    names = r["names"].split("|")
    masked = r["masked"]
    for i, n in enumerate(names):
        pos_name_counts[i][n] += 1
    for m in type_pat.finditer(masked):
        idx = int(m.group(1))
        ty = re.sub(r"'\w+\s*", "", m.group(2).strip()).strip()
        if idx < len(names):
            type_name_counts[ty][names[idx]] += 1

type_best = {ty: c.most_common(1)[0][0] for ty, c in type_name_counts.items()}
pos_best = {i: c.most_common(1)[0][0] for i, c in pos_name_counts.items()}

preds = {}
for r in test:
    n = int(r["num_vars"])
    masked = r["masked"]
    types_by_idx = {}
    for m in type_pat.finditer(masked):
        idx = int(m.group(1))
        ty = re.sub(r"'\w+\s*", "", m.group(2).strip()).strip()
        types_by_idx[idx] = ty
    row_preds = []
    for i in range(n):
        ty = types_by_idx.get(i)
        row_preds.append(type_best.get(ty, pos_best.get(i, "x")))
    preds[r["id"]] = "|".join(row_preds)

write_submission(preds)
print("wrote type-signature heuristic baseline (reference only, not a valid final submission)")


In [ ]:
# Submission 3 (intended): character-level encoder/decoder trained on the GPU.
# Vocabulary is built from the TRAINING data only.
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_items = [
    {"masked": r["masked"], "answers": r["names"].split("|"), "num_vars": int(r["num_vars"])}
    for r in train
]
test_items = [
    {"id": r["id"], "masked": r["masked"], "num_vars": int(r["num_vars"])}
    for r in test
]

CHARS = sorted(set("".join(it["masked"] for it in train_items)) | {" "})
CHAR2I = {c: i + 3 for i, c in enumerate(CHARS)}
PAD, BOS, EOS = 0, 1, 2
VOCAB_SIZE = len(CHARS) + 3

NAME_CHARS = sorted(set("".join(c for it in train_items for n in it["answers"] for c in n)))
NCHAR2I = {c: i + 3 for i, c in enumerate(NAME_CHARS)}
NVOCAB_SIZE = len(NAME_CHARS) + 3
I2NCHAR = {v: k for k, v in NCHAR2I.items()}

MAX_CTX = 600
MAX_NAME_LEN = 20


def encode_text(s):
    s = s[:MAX_CTX]
    return [CHAR2I.get(c, CHAR2I.get(" ")) for c in s]


def var_positions(masked, n_vars):
    positions = {i: [] for i in range(n_vars)}
    for m in re.finditer(r"VAR(\d+)", masked):
        idx = int(m.group(1))
        if idx in positions and m.start() < MAX_CTX:
            positions[idx].append(m.start())
    return positions


ENC_DIM = 96


class Encoder(nn.Module):
    """Character CNN: stacked dilated 1-D convolutions with residual
    connections and LayerNorm (receptive field ~125 characters around every
    position)."""

    def __init__(self, vocab, hidden=ENC_DIM, dilations=(1, 2, 4, 8, 16), kernel=5):
        super().__init__()
        self.emb = nn.Embedding(vocab, hidden, padding_idx=PAD)
        self.dropout = nn.Dropout(0.2)
        self.convs = nn.ModuleList(
            nn.Conv1d(hidden, hidden, kernel, padding=d * (kernel - 1) // 2, dilation=d)
            for d in dilations
        )
        self.norms = nn.ModuleList(nn.LayerNorm(hidden) for _ in dilations)

    def forward(self, x):
        h = self.dropout(self.emb(x))
        for conv, norm in zip(self.convs, self.norms):
            c = conv(h.transpose(1, 2)).transpose(1, 2)
            h = norm(h + self.dropout(F.gelu(c)))
        return h


class AttnPool(nn.Module):
    """Learned softmax attention over every occurrence position of a
    placeholder (declaration site, each use site), so the model weighs
    whichever sites are most informative for that variable."""

    def __init__(self, hidden_2x):
        super().__init__()
        self.score = nn.Linear(hidden_2x, 1)

    def forward(self, enc_out, ctx_positions):
        B, T, H = enc_out.shape
        P = max(1, max(len(p) for p in ctx_positions))
        idx = torch.zeros(B, P, dtype=torch.long)
        mask = torch.zeros(B, P, dtype=torch.bool)
        for i, positions in enumerate(ctx_positions):
            valid = [p for p in positions if p < T][:P]
            if valid:
                idx[i, :len(valid)] = torch.tensor(valid)
                mask[i, :len(valid)] = True
        idx = idx.to(enc_out.device)
        mask = mask.to(enc_out.device)
        reps = torch.gather(enc_out, 1, idx.unsqueeze(-1).expand(-1, -1, H))
        scores = self.score(reps).squeeze(-1).masked_fill(~mask, float("-inf"))
        scores = torch.where(mask.any(dim=1, keepdim=True), scores, torch.zeros_like(scores))
        weights = torch.softmax(scores, dim=1) * mask
        return (weights.unsqueeze(-1) * reps).sum(dim=1)


class Decoder(nn.Module):
    def __init__(self, vocab, hidden=96, ctx_dim=ENC_DIM):
        super().__init__()
        self.emb = nn.Embedding(vocab, hidden, padding_idx=PAD)
        self.rnn = nn.GRU(hidden + ctx_dim, hidden, batch_first=True)
        self.out_proj = nn.Linear(hidden, vocab)
        self.ctx_proj = nn.Linear(ctx_dim, hidden)

    def forward(self, ctx_vec, target_ids=None, max_len=MAX_NAME_LEN):
        B = ctx_vec.size(0)
        h = torch.tanh(self.ctx_proj(ctx_vec)).unsqueeze(0)
        if target_ids is not None:
            inp = torch.full((B, 1), BOS, dtype=torch.long, device=ctx_vec.device)
            inp = torch.cat([inp, target_ids[:, :-1]], dim=1)
            e = self.emb(inp)
            ctx_rep = ctx_vec.unsqueeze(1).expand(-1, e.size(1), -1)
            out, _ = self.rnn(torch.cat([e, ctx_rep], dim=-1), h)
            return self.out_proj(out)
        outs = []
        cur = torch.full((B, 1), BOS, dtype=torch.long, device=ctx_vec.device)
        hcur = h
        for _ in range(max_len):
            e = self.emb(cur)
            o, hcur = self.rnn(torch.cat([e, ctx_vec.unsqueeze(1)], dim=-1), hcur)
            nxt = self.out_proj(o[:, -1, :]).argmax(-1, keepdim=True)
            outs.append(nxt)
            cur = nxt
        return torch.cat(outs, dim=1)


enc = Encoder(VOCAB_SIZE).to(device)
attn_pool = AttnPool(ENC_DIM).to(device)
dec = Decoder(NVOCAB_SIZE).to(device)
opt = torch.optim.Adam(
    list(enc.parameters()) + list(attn_pool.parameters()) + list(dec.parameters()), lr=2e-3
)


def pool_context(enc_out, ctx_positions):
    return attn_pool(enc_out, ctx_positions)


def make_batch(batch_items):
    xs, ctx_positions, names = [], [], []
    for it in batch_items:
        ids = encode_text(it["masked"])
        pos_map = var_positions(it["masked"], it["num_vars"])
        for slot, name in enumerate(it["answers"]):
            positions = pos_map.get(slot, [])
            if not positions:
                continue
            xs.append(ids)
            ctx_positions.append(positions)
            names.append(name[:MAX_NAME_LEN])
    if not xs:
        return None
    maxlen = max(len(x) for x in xs)
    xpad = torch.zeros(len(xs), maxlen, dtype=torch.long, device=device)
    for i, x in enumerate(xs):
        xpad[i, :len(x)] = torch.tensor(x, device=device)
    return xpad, ctx_positions, names


def names_to_tensor(names):
    maxlen = max(len(n) for n in names) + 1
    t = torch.zeros(len(names), maxlen, dtype=torch.long, device=device)
    for i, n in enumerate(names):
        for j, c in enumerate(n):
            t[i, j] = NCHAR2I.get(c, NCHAR2I.get("_", 3))
        t[i, len(n)] = EOS
    return t, maxlen


BATCH = 64
EPOCHS = 20
n_batches = (len(train_items) + BATCH - 1) // BATCH

for epoch in range(EPOCHS):
    random.shuffle(train_items)
    total_loss, n = 0.0, 0
    enc.train(); attn_pool.train(); dec.train()
    for bi in range(n_batches):
        batch = train_items[bi * BATCH:(bi + 1) * BATCH]
        made = make_batch(batch)
        if made is None:
            continue
        xpad, ctx_positions, names = made
        enc_out = enc(xpad)
        ctx_vec = pool_context(enc_out, ctx_positions)
        target, maxlen = names_to_tensor(names)
        logits = dec(ctx_vec, target_ids=target, max_len=maxlen)
        loss = F.cross_entropy(logits.reshape(-1, NVOCAB_SIZE), target.reshape(-1), ignore_index=PAD)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item(); n += 1
    print(f"epoch {epoch} loss {total_loss / max(n,1):.4f}")


In [ ]:
# Evaluate the trained model on the test set and write the final submission.
enc.eval(); attn_pool.eval(); dec.eval()
EVAL_BATCH = 64
preds = {}
with torch.no_grad():
    for bi in range(0, len(test_items), EVAL_BATCH):
        batch = test_items[bi:bi + EVAL_BATCH]
        ids_list = [encode_text(it["masked"]) for it in batch]
        maxlen = max(len(x) for x in ids_list)
        xpad = torch.zeros(len(batch), maxlen, dtype=torch.long, device=device)
        for i, x in enumerate(ids_list):
            xpad[i, :len(x)] = torch.tensor(x, device=device)
        enc_out = enc(xpad)

        max_n = max(it["num_vars"] for it in batch)
        for slot in range(max_n):
            ctx_positions, relevant = [], []
            for i, it in enumerate(batch):
                positions = var_positions(it["masked"], it["num_vars"]).get(slot, [])
                if positions:
                    ctx_positions.append(positions)
                    relevant.append(i)
            if not relevant:
                continue
            ctx_vec = pool_context(enc_out[relevant], ctx_positions)
            out_ids = dec(ctx_vec, target_ids=None)
            for j, i in enumerate(relevant):
                chars = []
                for cid in out_ids[j].tolist():
                    if cid in (EOS, PAD):
                        break
                    chars.append(I2NCHAR.get(cid, ""))
                batch[i].setdefault("_preds", {})[slot] = "".join(chars)

        for it in batch:
            n = it["num_vars"]
            name_seq = [it.get("_preds", {}).get(s, "") for s in range(n)]
            preds[it["id"]] = "|".join(name_seq)

write_submission(preds)
print("wrote trained-reference submission")
